In [1]:
import sys; sys.path.insert(0, "../src")
import numpy as np
import forecast as F

In [2]:
df = F.load_all()
X = F.make_features(df); y = df["demand"]
X["demand_lag_1d"] = df["demand"].shift(48)        # NOT available in real use
ok = X.notna().all(axis=1); X, y = X[ok], y[ok]
train, test = X.index < F.TEST_START, X.index >= F.TEST_START
feats = F.CALENDAR + F.LAGS + F.WEATHER
mae = lambda m, f, mask: np.abs(y[mask] - m.predict(X.loc[mask, f])).mean()

In [3]:
# 1. Honest setup
m = F.fit(X.loc[train, feats], y[train], 500); print("honest:", mae(m, feats, test))

honest: 1006.7427199018709


In [4]:
# 2. Leak: add yesterday's demand
f2 = feats + ["demand_lag_1d"]
m = F.fit(X.loc[train, f2], y[train], 500); print("1-day lag:", mae(m, f2, test))

1-day lag: 956.414098506137


In [5]:
# 3. Leak: random split instead of time split
r = np.random.default_rng(0).random(len(X)) < 0.8
m = F.fit(X.loc[r, feats], y[r], 500); print("random split:", mae(m, feats, ~r))

random split: 645.4726969289347
